# Module 8: Functions

**Utrains Python Fundamentals** &middot; lab notebook

*Package reusable logic, pass arguments four different ways, and stream results with yield.*

## By the end of this notebook you can

- Define a function with def and call it
- Tell a parameter from an argument, and know why return ends a function
- Use all four kinds of argument, including *args and **kwargs
- Write a generator with yield and see why streaming needs it
- Pass a function itself as a value into another function

## How to work through it

It follows the Module 8 slide deck, slide by slide.

The headings below are the slide numbers from the deck. The explanation for
each one is on the slide and in the [README](../README.md); this notebook is
where you run the code.

Run every cell in order with **Shift + Enter**.

Two cells are marked **Your turn**. They contain `____` where a piece of the
syntax is missing, so they fail if you run them as they are. That is
deliberate. Replace each `____`, then run the cell until it succeeds.

The last section is the **Lab**: a short task with no code written for you.

**Assumed knowledge.** Modules 1 to 7. You have already read functions in Module 7's retry example; this is where you write them.

## Slide 2 &middot; What Is a Function?

In [ ]:
def greet():
    print("Hello! Welcome to Python.")


greet()          # calls the function
greet()

## Slide 3 &middot; Parameters, Arguments, Return

In [ ]:
def greet(name):
    print(f"Hello, {name}!")


greet("Alice")


def square(num):
    return num * num


result = square(4)
print("Square:", result)

In [ ]:
def early_exit():
    return "first"
    print("this line never runs")


print(early_exit())

## Slide 4 &middot; Four Kinds of Arguments

In [ ]:
def add(a, b):
    return a + b


print(add(3, 5))

## Slide 5 &middot; Default and Keyword Arguments

In [ ]:
def greet(name="Guest"):
    print(f"Hello, {name}!")


greet()
greet("Alice")


def describe_pet(animal, name):
    print(f"{name} is a {animal}.")


describe_pet(animal="dog", name="Buddy")
describe_pet(name="Kitty", animal="cat")

---

### Your turn 1

Write a deployment message builder. The service is passed by position, the environment has a default of `"dev"`, and the message is handed back to the caller rather than printed inside the function.

Replace each `____` below, then run the cell. It will not run until you do.

In [ ]:
# TODO: give environment a default of "dev", and send the message back.
def deployment_message(service, environment____):
    ____ f"Deploying {service} to {environment}"


print(deployment_message("api-gateway"))
print(deployment_message("api-gateway", environment="prod"))

## Slide 6 &middot; *args and **kwargs

In [ ]:
def add_numbers(*args):
    print("received a", type(args).__name__, ":", args)
    return sum(args)


print(add_numbers(1, 2, 3, 4))

In [ ]:
def describe_person(**kwargs):
    print("received a", type(kwargs).__name__)
    for key, value in kwargs.items():
        print(f"  {key}: {value}")


describe_person(name="Alice", age=25, city="New York")

## Slide 7 &middot; Generators: yield Instead of return

In [ ]:
def stream_reply(tokens):
    full = ""
    for token in tokens:
        full += token
        yield full


for partial in stream_reply(["Hi, ", "I ", "am ", "an ", "AI."]):
    print(partial)

## Slide 8 &middot; Passing a Function as a Value

In [ ]:
def run_chat(fn, user_text):
    return fn(user_text, history=[])


def my_bot(message, history):
    return f"You said: {message}"


print(run_chat(fn=my_bot, user_text="Hello"))
# note: pass my_bot, not my_bot()

---

### Your turn 2

Write a generator that yields incident status updates one at a time, then a runner that takes any function as a value and applies it to a message.

Replace each `____` below, then run the cell. It will not run until you do.

In [ ]:
updates = ["triage started", "root cause found", "fix deployed", "resolved"]


def status_stream(items):
    for item in items:
        # TODO: hand back one value at a time WITHOUT ending the function.
        ____ f"[update] {item}"


for line in status_stream(updates):
    print(line)


def process(callback, message):
    # TODO: call whatever function was handed in.
    return ____(message)


def shout(text):
    return text.upper()


# TODO: pass the function itself, not its result.
print(process(____, "incident closed"))

---

# More use cases

The same ideas, applied to situations you will meet in real work. Run each one, then change a value and run it again.

## Use case 1 &middot; AI &middot; Build a call payload with **kwargs

In [ ]:
def build_request(model, prompt, **options):
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
    }
    payload.update(options)
    return payload


basic = build_request("gpt-4o-mini", "Say hello.")
tuned = build_request(
    "gpt-4o-mini",
    "Say hello.",
    temperature=0.2,
    max_tokens=100,
    stream=True,
)

print(basic)
print()
for key, value in tuned.items():
    print(f"  {key}: {value}")

## Use case 2 &middot; AI &middot; Stream tokens and keep a running total

In [ ]:
def stream(tokens):
    so_far = ""
    for token in tokens:
        so_far += token
        yield so_far


pieces = ["The ", "target ", "group ", "is ", "unhealthy."]

count = 0
for partial in stream(pieces):
    count += 1
    print(f"  {count:>2}  {partial}")

print()
print("tokens streamed:", count)
print("final answer   :", partial)

## Use case 3 &middot; AI &middot; Let the caller decide how to format

In [ ]:
def plain(role, text):
    return f"{role}: {text}"


def bracketed(role, text):
    return f"[{role.upper():>9}] {text}"


def render(formatter, history):
    for role, text in history:
        print(formatter(role, text))


history = [("user", "Why is it slow?"), ("assistant", "Two targets are down.")]

render(plain, history)
print()
render(bracketed, history)

## Use case 4 &middot; A retry helper worth reusing

In [ ]:
def call_with_retry(name, attempts=3, succeed_on=2):
    for attempt in range(1, attempts + 1):
        if attempt >= succeed_on:
            return f"{name} succeeded on attempt {attempt}"
        print(f"  {name}: attempt {attempt} failed")
    return f"{name} gave up after {attempts} attempts"


print(call_with_retry("health-check"))
print(call_with_retry("slow-service", attempts=5, succeed_on=4))
print(call_with_retry("broken-service", attempts=2, succeed_on=99))

---

## Lab: A small cost calculator toolkit

Write three functions that work together, one for each of the ideas in this
module.

`monthly_cost(*resources)` takes any number of per-resource monthly costs and
returns the total.

`format_bill(total, currency="USD", warn_above=500)` returns a formatted string
with the total to two decimal places, and appends `" OVER BUDGET"` when the
total is above the threshold. Both extra parameters must have working defaults.

`report(builder, *resources)` takes a **function** as its first argument, calls
`monthly_cost` on the resources, passes the total to `builder`, and returns the
result.

Prove all three work by calling `report(format_bill, 120.0, 340.5, 88.25)`.

For extra credit, add a generator that yields each resource cost as a running
total, the way `stream_reply` did.

**Done when:**

- [ ] monthly_cost uses *args and works with any number of values
- [ ] format_bill has two working default arguments
- [ ] report accepts a function as a value and calls it
- [ ] The final call prints a formatted total with the budget warning

Write your answer in the cell below. There is no starter code on purpose.

In [ ]:
# Your lab answer goes here.

---

## Practice exercises

The four exercises from the module's practice slide are in the
[README](../README.md#practice-exercises) and repeated on the slide. There are
4 of them. Do them in a scratch cell here or in a `.py` file.

## Module complete

You can now write functions, handle arguments four ways, and stream results with yield.

*Utrains &middot; support@utrains.org &middot; https://utrains.org*